In [2]:

import glob
import numpy as np
import nibabel as nib

In [3]:
fdir = "./data/after/rois"

In [4]:

def get_center(img):
    d = img.get_fdata()
    xs, ys, zs = np.where(d)
    return {"x": xs.mean(), "y": ys.mean(), "z": zs.mean()}


In [14]:

def load_imgs(tname):
    fnames = {}
    imgs = {}
    # if "CC" in tname: 
    #     fnames["lpi"] = glob.glob(f"{fdir}/{tname}*LPI*")[0]
    #     fnames["ras"] = glob.glob(f"{fdir}/{tname}*RAS*")[0]
    #     print(fnames)
    #     imgs["lpi"] = nib.load(fnames["lpi"])
    #     imgs["ras"] = nib.load(fnames["ras"])
    #     if len(imgs) < 2: 
    #         raise Exception(f"missing data for {tname} (only loaded {list(imgs.keys())}). Aborted.")
        
    # else: 
    fnames["llpi"] = glob.glob(f"{fdir}/left{tname}*LPI*")[0]
    fnames["lras"] = glob.glob(f"{fdir}/left{tname}*RAS*")[0]
    fnames["rlpi"] = glob.glob(f"{fdir}/right{tname}*LPI*")[0]
    fnames["rras"] = glob.glob(f"{fdir}/right{tname}*RAS*")[0]
    print(fnames)
    imgs["llpi"] = nib.load(fnames["llpi"])
    imgs["lras"] = nib.load(fnames["lras"])
    imgs["rlpi"] = nib.load(fnames["rlpi"])
    imgs["rras"] = nib.load(fnames["rras"])
    if len(imgs) < 4: 
        raise Exception(f"missing data for {tname} (only loaded {list(imgs.keys())}). Aborted.")

    return imgs, fnames


In [17]:
"""
(orientation: small -> big)
x: R -> L
y: A -> P 
z: I -> S
"""

## define the major orientation of the tracts
tract_orientations = {
    "MDLFang":"z", 
    "MDLFspl":"z", 
    "Uncinate":"y", 
    "Aslant":"z"
}

for tname, orientation in tract_orientations.items():
    print(f"====={tname}=====")

    imgs, fnames = load_imgs(tname)

    centers = {tlab: get_center(img) for tlab, img in imgs.items()}
    for lab, cs in centers.items():
        print(lab, cs)

    ## sanity check: left x > right x
    print("=== Sanity check: left x > right x for all left/right pairs ===")
    for llab in ["lras", "llpi"]:
        for rlab in ["rras", "rlpi"]:
            if centers[llab]["x"] < centers[rlab]["x"]:
                raise Exception(f"{llab} {rlab} failed: data in different coordinates. Aborted.")
    print("passed")

    ## check each major orientation
    if orientation == "y": 
        ## y: A -> P  --> ras < lpi
        print(f"=== Checking ras y < lpi y (A -> P) ===")
        
        for raslab, lpilab in [["lras", "llpi"], ["rras", "rlpi"]]:
            print(raslab, lpilab)
            if centers[raslab][orientation] < centers[lpilab][orientation]:
                print("True")
            else: 
                print("swapping ras/lpi ")
                rasfn = fnames[raslab]
                lpifn = fnames[lpilab]
                tempfn = f"{fdir}/temp"
                print("reveresing fnames")
                # os.rename(rasfn, tempfn)
                # os.rename(lpifn, rasfn)
                # os.rename(tempfn, lpifn)
        
    if orientation == "z": 
        ## z: I -> S --> ras > lpi
        print(f"=== Checking ras z > lpi z (I -> S) ===")
        
        for raslab, lpilab in [["lras", "llpi"], ["rras", "rlpi"]]:
            print(raslab, lpilab)
            if centers[raslab][orientation] > centers[lpilab][orientation]:
                print("True")
            else: 
                print("swapping ras/lpi ")
                rasfn = fnames[raslab]
                lpifn = fnames[lpilab]
                tempfn = f"{fdir}/temp"
                print("reveresing fnames")
                # os.rename(rasfn, tempfn)
                # os.rename(lpifn, rasfn)
                # os.rename(tempfn, lpifn)


=====MDLFang=====
{'llpi': './data/after/rois/leftMDLFang_box_1mm_LPI_FiberEndpoint.nii.gz', 'lras': './data/after/rois/leftMDLFang_box_1mm_RAS_FiberEndpoint.nii.gz', 'rlpi': './data/after/rois/rightMDLFang_box_1mm_LPI_FiberEndpoint.nii.gz', 'rras': './data/after/rois/rightMDLFang_box_1mm_RAS_FiberEndpoint.nii.gz'}


llpi {'x': 138.71152296535053, 'y': 151.67526188557613, 'z': 103.2925060435133}
lras {'x': 147.3880308880309, 'y': 97.27413127413128, 'z': 57.4980694980695}
rlpi {'x': 52.03747870528109, 'y': 150.98126064735945, 'z': 105.06473594548552}
rras {'x': 47.43445692883895, 'y': 92.81086142322097, 'z': 52.37265917602996}
=== Sanity check: left x > right x for all left/right pairs ===
passed
=== Checking ras z > lpi z (I -> S) ===
lras llpi
swapping ras/lpi 
reveresing fnames
rras rlpi
swapping ras/lpi 
reveresing fnames
=====MDLFspl=====
{'llpi': './data/after/rois/leftMDLFspl_box_1mm_LPI_FiberEndpoint.nii.gz', 'lras': './data/after/rois/leftMDLFspl_box_1mm_RAS_FiberEndpoint.nii.gz', 'rlpi': './data/after/rois/rightMDLFspl_box_1mm_LPI_FiberEndpoint.nii.gz', 'rras': './data/after/rois/rightMDLFspl_box_1mm_RAS_FiberEndpoint.nii.gz'}
llpi {'x': 119.88125665601704, 'y': 158.41373801916933, 'z': 118.54472843450479}
lras {'x': 146.48939512961508, 'y': 95.08169677926159, 'z': 57.36292223095051}
rlpi 